Steps:
1. Import packages needed
2. Input data (includes train test split, normalize)
3. Create model with Hyperparameter analysis
4. Acquire parameters / Evaluate model
5. Visualizations (GridSearch Heatmap, Scatterplot with best Gridsearch)

## 1. Import Packages

In [1]:
import matplotlib.pyplot as plt
import altair as alt
import numpy as np
import pandas as pd

In [2]:
np.set_printoptions(precision=5)
random_state = 42
from ast import literal_eval

import sklearn
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.manifold import TSNE
import umap
from sklearn.metrics import pairwise_distances
print("Scikit-learn: {}".format(sklearn.__version__))

Scikit-learn: 1.8.0


## 2. Input Data

In [3]:
df = pd.read_csv('./data/final-scryfall-unique-artwork.csv')
creature_df = df[df['type'].str.contains('Creature')]
creature_df.head()

,id,oracle_id,name,set_id,set,set_name,artist_ids,artist,released_at,type_line,...,power,toughness,edhrec_rank,type,subtype,legality_commander,legality_standard,price_usd,image_uri_normal,image_uri_art_crop
1,0000579f-7b35-4ed3-b44c-db2a538066fe,44623693-51d6-49ad-8cd7-140505caf02f,Fury Sliver,c1d109bc-ffd8-428f-8d7d-3f8d7e648046,tsp,Time Spiral,['d48dd097-720d-476a-8722-6a02854ae28b'],Paolo Parente,2006-10-06,Creature — Sliver,...,3,3,9808.0,Creature,Sliver,legal,not_legal,0.46,https://cards.scryfall.io/normal/front/0/0/000...,https://cards.scryfall.io/art_crop/front/0/0/0...
2,00006596-1166-4a79-8443-ca9f82e6db4e,8ae3562f-28b7-4462-96ed-be0cf7052ccc,Kor Outfitter,eb16a2bd-a218-4e4e-8339-4aa1afc0c8d2,zen,Zendikar,['aa7e89ed-d294-4633-9057-ce04dacfcfa4'],Kieran Yanner,2009-10-02,Creature — Kor Soldier,...,2,2,19672.0,Creature,Kor Soldier,legal,not_legal,0.11,https://cards.scryfall.io/normal/front/0/0/000...,https://cards.scryfall.io/art_crop/front/0/0/0...
3,0000cd57-91fe-411f-b798-646e965eec37,9f0d82ae-38bf-45d8-8cda-982b6ead1d72,Siren Lookout,fe0dad85-54bc-4151-9200-d68da84dd0f2,xln,Ixalan,['a8e7b854-b15a-421a-b66d-6e68187ae285'],Chris Rallis,2017-09-29,Creature — Siren Pirate,...,1,2,18843.0,Creature,Siren Pirate,legal,not_legal,0.04,https://cards.scryfall.io/normal/front/0/0/000...,https://cards.scryfall.io/art_crop/front/0/0/0...
4,0001f1ef-b957-4a55-b47f-14839cdbab6f,ef027846-be81-4959-a6b5-56bd01b1e68a,Venerable Knight,a90a7b2f-9dd8-4fc7-9f7d-8ea2797ec782,eld,Throne of Eldraine,['9c201dbe-db56-429a-87e6-189ea70c2632'],Colin Boyer,2019-10-04,Creature — Human Knight,...,2,1,18404.0,Creature,Human Knight,legal,not_legal,0.15,https://cards.scryfall.io/normal/front/0/0/000...,https://cards.scryfall.io/art_crop/front/0/0/0...
6,0002ab72-834b-4c81-82b1-0d2760ea96b0,645b5784-a6f7-4cf3-966a-e1a51420b96b,Mystic Skyfish,bc94aba1-7376-4e02-a12d-3a2efb66ab0f,m21,Core Set 2021,['bb677b1a-ce51-4888-83d6-5a94de461ff9'],Alayna Danner,2020-07-03,Creature — Fish,...,3,1,23732.0,Creature,Fish,legal,not_legal,0.10,https://cards.scryfall.io/normal/front/0/0/000...,https://cards.scryfall.io/art_crop/front/0/0/0...


In [4]:
# calculate thresholds for top 1%, bottom 1%, and middle 1% by price (acquired from MDS artowrk file)
creature_df.loc[:,'price_usd'] = creature_df['price_usd'].round(2)

top_1 = creature_df['price_usd'].quantile(0.99)
bottom_1 = creature_df['price_usd'].quantile(0.01)
mid_lower = creature_df['price_usd'].quantile(0.495) # just below median
mid_upper = creature_df['price_usd'].quantile(0.505) # just above median

top_1_df = creature_df[creature_df['price_usd'] >= top_1]
top_1_df['percentile'] = 'Top 1%'

bottom_1_df = creature_df[creature_df['price_usd'] <= bottom_1]
bottom_1_df['percentile'] = 'Bottom 1%'

mid_1_df = creature_df[(creature_df['price_usd'] >= mid_lower) & (creature_df['price_usd'] < mid_upper)]
mid_1_df['percentile'] = 'Middle 1%'

percentile_df = pd.concat([top_1_df, bottom_1_df, mid_1_df]).reset_index()

C:\Users\jaymj\AppData\Local\Temp\ipykernel_23936\2941743629.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_1_df['percentile'] = 'Top 1%'
C:\Users\jaymj\AppData\Local\Temp\ipykernel_23936\2941743629.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bottom_1_df['percentile'] = 'Bottom 1%'
C:\Users\jaymj\AppData\Local\Temp\ipykernel_23936\2941743629.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value in

In [5]:
#one hot encoding of labels (acquired from MDS artowrk file)
one_hot_encoding_df = percentile_df.copy()
one_hot_encoding_df['subtype'] = percentile_df['subtype'].fillna('').str.split()

mlb = MultiLabelBinarizer()
encoded_subtypes = mlb.fit_transform(one_hot_encoding_df['subtype'])
subtypes_df = pd.DataFrame(encoded_subtypes, columns='s_'+mlb.classes_, index=one_hot_encoding_df.index)

encoded_keywords = mlb.fit_transform(one_hot_encoding_df['keywords'].apply(literal_eval))
keywords_df = pd.DataFrame(encoded_keywords, columns='k_'+mlb.classes_, index=one_hot_encoding_df.index)

encoded_colors = mlb.fit_transform(one_hot_encoding_df['color_identity'].apply(literal_eval))
colors_df = pd.DataFrame(encoded_colors, columns='c_'+mlb.classes_, index=one_hot_encoding_df.index)

features = pd.concat([subtypes_df, keywords_df, colors_df], axis=1)

## 3. Create model

In [6]:
#Creating coordinates for T-SNE model with hyperparameter adjustment
tsne_params = {
    'perplexity' : [5, 10, 25, 40, 50],
    'learning_rate' : [10, 50, 100, 250, 500, 1000, 'auto'],
    'metric' : ['euclidean', 'mahattan', 'cosine'],
    'init' : ['pca','random'],
    }

tsne_clf = TSNE(n_components = 2, init='pca', random_state=random_state)

tsne_pos = tsne_clf.fit(features).embedding_
tsne_percentile_df = percentile_df.copy()
tsne_percentile_df['x'] = [x[0] for x in tsne_pos]
tsne_percentile_df['y'] = [y[1] for y in tsne_pos]


In [8]:
#Creating coordinates for U-MAP model
umap_params = parameters = {
    'n_neighbors' : [2, 5, 8, 10, 15, 25, 50, 100],
    'min_distance' : [0, 0.005, 0.01, 0.1 , 0.25, 0.5, 0.75, 1.0],
    'metric' : ['euclidean', 'mahattan', 'cosine'],
    }

umap_clf = umap.UMAP(random_state=random_state)
umap_pos = umap_clf.fit_transform(features)
umap_percentile_df = percentile_df.copy()
umap_percentile_df['x'] = [x[0] for x in umap_pos]
umap_percentile_df['y'] = [y[1] for y in umap_pos]

c:\Users\jaymj\anaconda3\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


## 5. Visualizations

In [13]:
def genPlot(df, sample_n=200, art_crop=True):
    # input: df -- dataframe (augmented with the x/y columns)
    # input: sample_n -- number of cards to display from each percentile, randomly sampled
    # input: art_crop -- True to use cropped art image, False to use full card image
    # return: an altair chart (e.g., return alt.Chart(...))

    sampled_df = df.groupby('percentile').apply(lambda x: x.sample(n=sample_n, random_state=42)).rename(columns={'percentile':'sample_percentile'}).reset_index()

    image_url = 'image_uri_art_crop' if art_crop else 'image_uri_normal'
    
    images = alt.Chart(sampled_df).mark_image(width=40, height=40).encode(
        x=alt.X('x', axis=None),
        y=alt.Y('y', axis=None),
        url=image_url,
        tooltip=[
            alt.Tooltip("name"),
            alt.Tooltip("subtype"),
            alt.Tooltip("keywords"),
            alt.Tooltip("color_identity"),
            alt.Tooltip("price_usd")
        ]
    ).properties(
        width=1000,
        height=1000
    )

    squares = alt.Chart(sampled_df).mark_square(size=2000).encode(
        x=alt.X('x', axis=None),
        y=alt.Y('y', axis=None),
        color=alt.Color('percentile:N', legend=alt.Legend(title="Percentile"))
    )

    return squares + images

In [14]:
#tsne visualization
tsne_chart = genPlot(tsne_percentile_df, 50)
tsne_chart

alt.LayerChart(...)

In [15]:
#UMAP visualization
umap_chart = genPlot(umap_percentile_df, 50)
umap_chart

alt.LayerChart(...)